# InsurPrime: Can You Guess the Insurance Premium?
by Crédit Agricole Assurances

## Model: CatBoost Regressor

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import TweedieRegressor
from catboost import CatBoostRegressor

In [4]:
from feature_eng import add_engineered_features
from data_preprocessing import preprocess_data

## Data Loading

In [5]:
X_train=pd.read_csv("train_input.csv")
y_train=pd.read_csv("train_output.csv")
X_test=pd.read_csv("test_input.csv")

C:\Users\nelso\AppData\Local\Temp\ipykernel_17800\3283101309.py:1: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  X_train=pd.read_csv("train_input.csv")
C:\Users\nelso\AppData\Local\Temp\ipykernel_17800\3283101309.py:3: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  X_test=pd.read_csv("test_input.csv")


## Data preprocessing

In [6]:
X_train_clean, y_train_clean, fitted = preprocess_data(
    X_train, 
    y_train, 
    is_train=True
)

X_test_clean, _, _ = preprocess_data(
    X_test, 
    is_train=False,
    fitted_transformers=fitted
)

In [7]:
X_train_clean, y_train_clean, fitted = preprocess_data(
    X_train, 
    y_train, 
    is_train=True
)

X_test_clean, _, _ = preprocess_data(
    X_test, 
    is_train=False,
    fitted_transformers=fitted
)

## Adding Features

In [10]:
# train
X_train_fe, fe_meta = add_engineered_features(X_train_clean, is_train=True)

# test
X_test_fe, _ = add_engineered_features(X_test_clean, is_train=False, feature_meta=fe_meta)

### CatBoost Regressor

In [21]:
## Combine train + test for consistent encoding

n_train = len(X_train_fe)
X_combined = pd.concat([X_train_fe, X_test_fe], axis=0, ignore_index=True)

# categorical columns
cat_cols = X_combined.select_dtypes(include=["object", "category"]).columns.tolist()

# Ordinal encode cat cols
# Cast to string so NaNs become 'nan' and are treated as a level
if len(cat_cols) > 0:
    ord_enc = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
    X_combined[cat_cols] = ord_enc.fit_transform(X_combined[cat_cols].astype(str)).astype(int)

# everything is numeric
X_train_enc = X_combined.iloc[:n_train].copy()
X_test_enc  = X_combined.iloc[n_train:].copy()

# Indices of categorical features
cat_indices = [X_train_enc.columns.get_loc(c) for c in cat_cols]


## Train/val

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_enc,
    y_train_clean,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

# Exposure (ANNEE_ASSURANCE) as sample_weight and for charge computation
w_tr  = pd.to_numeric(X_tr["ANNEE_ASSURANCE"], errors="coerce").fillna(0).clip(lower=0)
w_val = pd.to_numeric(X_val["ANNEE_ASSURANCE"], errors="coerce").fillna(0).clip(lower=0)

In [24]:
## frequency model = > Poisson

y_tr_freq = y_tr["FREQ"].astype(float)

# Parameters
params_freq = {
    "min_data_in_leaf": 50,
    "iterations": 2000,
    "learning_rate": 0.05,
    "l2_leaf_reg": 5.0,
    "loss_function": "Poisson",
    "verbose": 0,
}

freq_model = CatBoostRegressor(
    **params_freq,
    cat_features=cat_indices,
    random_seed=42
)

freq_model.fit(X_tr, y_tr_freq, sample_weight=w_tr)

pred_freq_val = np.clip(freq_model.predict(X_val), 0, None)

KeyboardInterrupt: 

In [ ]:
## Severity model => Gamma (approximated by Tweedie:variance_power=1.5 in CatBoost)

mask_claim_tr = y_tr["CHARGE"] > 0
X_tr_sev = X_tr[mask_claim_tr].copy()
y_tr_sev = y_tr.loc[mask_claim_tr, "CM"].astype(float)
y_tr_sev = np.maximum(y_tr_sev, 1.0)  # Tweedie needs y >= 0

params_sev = {
    "min_data_in_leaf": 150,
    "iterations": 1000,
    "learning_rate": 0.01,
    "l2_leaf_reg": 5.0,
    "loss_function": "Tweedie:variance_power=1.5",
    "verbose": 0,
}

sev_model = CatBoostRegressor(
    **params_sev,
    cat_features=cat_indices,
    random_seed=42
)
sev_model.fit(X_tr_sev, y_tr_sev)

pred_sev_val = sev_model.predict(X_val)


In [ ]:
# Clip severity
limit_sev_val = y_tr_sev.quantile(0.999) * 2.0
pred_sev_val = np.clip(pred_sev_val, 0, limit_sev_val)


## Val CHARGE and RMSE

pred_charge_val = pred_freq_val * pred_sev_val * w_val

rmse_val = np.sqrt(mean_squared_error(y_val["CHARGE"], pred_charge_val))
print(f"Validation RMSE: {rmse_val:,.2f}")

In [ ]:
test_ids = X_test["ID"].copy()
test_exposure = X_test["ANNEE_ASSURANCE"]

In [ ]:
## Train on full train and predict on test

print("\n Training ensemble on FULL data and scoring test ")

w_full  = pd.to_numeric(X_train_enc["ANNEE_ASSURANCE"], errors="coerce").fillna(0).clip(lower=0)
y_full_freq = y_train_clean["FREQ"].astype(float)

mask_claim_full = y_train_clean["CHARGE"] > 0
X_full_sev      = X_train_enc[mask_claim_full].copy()
y_full_sev      = y_train_clean.loc[mask_claim_full, "CM"].astype(float)
y_full_sev      = np.maximum(y_full_sev, 1.0)

n_bags = 5
preds_freq_test = np.zeros(len(X_test_enc))
preds_sev_test  = np.zeros(len(X_test_enc))

for i in range(n_bags):
    seed = 42 + i
    print(f"Bag {i+1}/{n_bags}...")

    # Frequency
    model_freq = CatBoostRegressor(
        loss_function="Poisson", # Changed from loss
        cat_features=cat_indices, # Changed from categorical_features
        random_seed=seed, # Changed from random_state
        **params_freq
    )
    model_freq.fit(X_train_enc, y_full_freq, sample_weight=w_full)
    preds_freq_test += model_freq.predict(X_test_enc)

    # Severity
    model_sev = CatBoostRegressor(
        loss_function="Tweedie:variance_power=1.5", # Changed from loss
        cat_features=cat_indices, # Changed from categorical_features
        random_seed=seed, # Changed from random_state
        **params_sev
    )
    model_sev.fit(X_full_sev, y_full_sev)
    preds_sev_test += model_sev.predict(X_test_enc)

# Average the bagged predictions
preds_freq_test /= n_bags
preds_sev_test  /= n_bags

# Clip predictions
preds_freq_test = np.clip(preds_freq_test, 0, None)
limit_sev_full = y_full_sev.quantile(0.999) * 2.0
preds_sev_test = np.clip(preds_sev_test, 0, limit_sev_full)

# Final charge = freq * severity * exposure
test_exposure_clean = pd.to_numeric(test_exposure, errors="coerce").fillna(0).clip(lower=0)
pred_charge_test = preds_freq_test * preds_sev_test * test_exposure_clean.to_numpy(dtype=float)


## submission

submission = pd.DataFrame({
    "ID": test_ids,
    "FREQ": preds_freq_test,
    "CM": preds_sev_test,
    "ANNEE_ASSURANCE": test_exposure_clean,
    "CHARGE": pred_charge_test,
})

os.makedirs("Submissions", exist_ok=True)
submission.to_csv("Submissions/submission_hgbr.csv", index=False)
print("\nSaved submission_hgbr.csv")
print(submission.head())

### Useful graphs to understand the model

In [ ]:
df_val = pd.DataFrame({
    "charge_true": y_val["CHARGE"].values,
    "charge_pred": pred_charge_val,
    "freq_pred":   pred_freq_val,
    "sev_pred":    pred_sev_val,
    "exposure":    w_val.values,
})

# Derived columns
df_val["has_claim"]   = (df_val["charge_true"] > 0).astype(int)
df_val["sev_true"]    = np.where(df_val["has_claim"] == 1,
                                 df_val["charge_true"] / df_val["exposure"].replace(0, np.nan),
                                 np.nan)
df_val["log_true"]    = np.log1p(df_val["charge_true"])
df_val["log_pred"]    = np.log1p(df_val["charge_pred"])
df_val["log_resid"]   = df_val["log_true"] - df_val["log_pred"]